<a href="https://colab.research.google.com/github/juliotrecen/Analise-CPEA-vs-IFBOI/blob/main/An%C3%A1lise_CPEA_vs_IFBOI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐂 Apostila Técnica: Análise de Previsibilidade Pecuária

Este material visa validar o **IFBOI** como indicador antecedente do **CEPEA**, utilizando rigor estatístico para evitar armadilhas comuns em dados financeiros.

### Por que não usar a Correlação de Pearson Comum?
A correlação simples (Pearson) mede a relação linear entre dois níveis de preço. Em séries temporais, isso é **perigoso** devido ao fenômeno da **Correlação Espúria**. Séries financeiras geralmente possuem tendências (inflação, ciclo do gado). Se duas séries crescem ao longo do tempo, elas terão correlação alta mesmo que não tenham relação causal nenhuma.

Para um estudo sério, precisamos focar na **Variação (Retorno)** e na **Causalidade**, não apenas no movimento visual das curvas.

### O que é Correlação Espúria?

Uma **Correlação Espúria** ocorre quando duas variáveis parecem estar relacionadas, mas essa relação é puramente matemática ou coincidência, sem nenhuma conexão causal real.

#### O Exemplo Clássico
Imagine um gráfico que mostra o *aumento das vendas de sorvete* e o *aumento de ataques de tubarão*. A correlação será altíssima. Significa que sorvetes atraem tubarões? **Não.** Existe uma terceira variável oculta (o Verão). No calor, as pessoas tomam mais sorvete E entram mais no mar.

#### Em Séries Temporais (Financeiro)
Séries financeiras como o IFBOI e o CEPEA são frequentemente influenciadas por:
1. **Inflação**: Tudo tende a subir de preço nominal ao longo de décadas.
2. **Sazonalidade**: Ciclos de safra e entressafra que afetam todos os ativos do mesmo setor ao mesmo tempo.

Se usarmos a correlação simples, o software dirá que a relação é de 99%, mas isso pode ser apenas porque ambos os preços estão subindo devido à inflação. Por isso, usamos o **Teste ADF** e as **Diferenças (Retornos)**: ao subtrair o preço de hoje pelo de ontem, eliminamos a tendência de longo prazo e focamos apenas no 'pulso' do mercado. Se os 'pulsos' ainda estiverem sincronizados, aí sim temos uma relação real.

## Etapa 1: Ingestão e Preparação de Dados

**Por que esta etapa?**
Dados de fontes diferentes (B3 e CEPEA) possuem calendários e formatos distintos. Precisamos garantir que estamos comparando o preço do IFBOI e do CEPEA exatamente no mesmo dia, removendo ruídos de importação.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import statsmodels.api as sm

# 1. Carregamento e Sincronização
df_ifboi = pd.read_excel('/content/sample_data/IFBOI.xlsx')
df_cepea = pd.read_excel('/content/sample_data/CEPEA.xls', skiprows=3, engine='calamine')

df_ifboi['Data_Referencia'] = pd.to_datetime(df_ifboi['Data_Referencia'], errors='coerce')
df_cepea['Data'] = pd.to_datetime(df_cepea['Data'], format='%d/%m/%Y', errors='coerce')

df_master = pd.merge(
    df_ifboi[['Data_Referencia', 'Indice']],
    df_cepea[['Data', 'À vista R$']],
    left_on='Data_Referencia', right_on='Data', how='inner'
).drop(columns=['Data'])

# 2. Resumo Visual Inicial
fig_init = go.Figure(data=[
    go.Table(
        header=dict(values=['Data', 'IFBOI (Pontos)', 'CEPEA (R$)'], fill_color='royalblue', font=dict(color='white', size=14)),
        cells=dict(values=[df_master.Data_Referencia.dt.strftime('%d/%m/%Y'), df_master.Indice.round(2), df_master['À vista R$'].round(2)],
                   fill_color='lavender', font=dict(size=12))
    )
])
fig_init.update_layout(title='Base de Dados Sincronizada (Amostra)', margin=dict(l=0, r=0, t=40, b=0), height=300)
fig_init.show()

## Etapa 2: Normalização e Visualização de Tendência

**Por que esta etapa?**
O IFBOI está na casa dos 3.000 pontos, enquanto o CEPEA está na casa dos 250 reais. Se plotarmos sem normalizar, o CEPEA parecerá uma linha reta no fundo do gráfico. Usamos o **MinMaxScaler** para colocar ambos entre 0 e 1, permitindo comparar a *forma* do movimento.

In [ ]:
# Normalização para Comparação de Performance
scaler = MinMaxScaler()
df_norm = df_master.copy()
df_norm[['Indice', 'À vista R$']] = scaler.fit_transform(df_master[['Indice', 'À vista R$']])

# Dashboard de Tendência e Dispersão
fig = make_subplots(rows=2, cols=2,
                    specs=[[{'colspan': 2}, None], [{}, {}]],
                    subplot_titles=('Movimentação Relativa IFBOI vs CEPEA (Normalizado)', 'Distribuição IFBOI', 'Distribuição CEPEA'))

# Linhas de Tendência
fig.add_trace(go.Scatter(x=df_norm['Data_Referencia'], y=df_norm['Indice'], name='IFBOI', line=dict(color='#1f77b4', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_norm['Data_Referencia'], y=df_norm['À vista R$'], name='CEPEA', line=dict(color='#d62728', width=2)), row=1, col=1)

# Histogramas
fig.add_trace(go.Histogram(x=df_master['Indice'], name='IFBOI Dist', marker_color='#1f77b4', opacity=0.7), row=2, col=1)
fig.add_trace(go.Histogram(x=df_master['À vista R$'], name='CEPEA Dist', marker_color='#d62728', opacity=0.7), row=2, col=2)

fig.update_layout(height=700, template='plotly_dark', title_text='ANÁLISE VISUAL DE CONVERGÊNCIA', showlegend=True)
fig.show()

## Etapa 3: Estacionariedade e Retornos Logarítmicos

### 1. O que são Retornos? (A 'Pulsação' do Mercado)
Em vez de olharmos para o preço cheio ($P_t$), usamos a variação percentual contínua (Retorno Logarítmico):
$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$
- **Por que fazemos isso?** Se o boi custa $R\$$ 100 ou $R\$$ 300, o que importa para o investidor é a variação percentual. Os retornos colocam tudo na mesma régua e ajudam a tornar a série comparável ao longo do tempo.

### 2. O Teste ADF (A série tem 'rumo' ou é 'estável'?)
Imagine uma pessoa caminhando em um parque:
- **Série Não Estacionária**: É uma pessoa caminhando para o Norte. A cada passo, ela está mais longe do início. Ela tem uma **tendência**.
- **Série Estacionária**: É uma pessoa dando passos aleatórios para frente e para trás, mas sempre ficando em torno do mesmo lugar. A média é constante.

**O Problema**: Séries financeiras geralmente têm 'rumo' (inflação/tendência). O Teste ADF (Augmented Dickey-Fuller) testa matematicamente se a série precisa ser 'estabilizada' antes da análise.

### 3. Entendendo o p-valor (A Regra da Moeda)
O **p-valor** é a nossa margem de erro ou 'chance de estarmos sendo enganados pelo acaso'.
- **p-valor alto (> 0.05)**: A oscilação parece aleatória demais ou ainda tem muita tendência. Não podemos confiar.
- **p-valor baixo (< 0.05)**: É o nosso 'selo de qualidade'. Significa que temos mais de 95% de certeza que aquele comportamento é real e estável o suficiente para ser analisado estatisticamente.

In [ ]:
# Cálculo de Volatilidade (Retornos Log)
df_master['Ret_IFBOI'] = np.log(df_master['Indice'] / df_master['Indice'].shift(1))
df_master['Ret_CEPEA'] = np.log(df_master['À vista R$'] / df_master['À vista R$'].shift(1))
df_master = df_master.dropna()

# Visualização de Volatilidade
fig_vol = go.Figure()
fig_vol.add_trace(go.Scatter(x=df_master['Data_Referencia'], y=df_master['Ret_IFBOI'], name='Volatilidade IFBOI', opacity=0.6))
fig_vol.add_trace(go.Scatter(x=df_master['Data_Referencia'], y=df_master['Ret_CEPEA'], name='Volatilidade CEPEA', opacity=0.6))
fig_vol.update_layout(title='Análise de Estacionariedade (Retornos Diários)', template='plotly_white', yaxis_title='Variação % Log')
fig_vol.show()

# Texto de Apoio
def check_adf(serie, nome):
    p = adfuller(serie)[1]
    status = 'ESTACIONÁRIA' if p < 0.05 else 'NÃO ESTACIONÁRIA'
    print(f'-> {nome}: p-valor={p:.4f} | Status: {status}')

print('VALIDAÇÃO ESTATÍSTICA (ADF):')
check_adf(df_master['Ret_IFBOI'], 'IFBOI (Retornos)')
check_adf(df_master['Ret_CEPEA'], 'CEPEA (Retornos)')

VALIDAÇÃO ESTATÍSTICA (ADF):
-> IFBOI (Retornos): p-valor=0.0000 | Status: ESTACIONÁRIA
-> CEPEA (Retornos): p-valor=0.0000 | Status: ESTACIONÁRIA


## Etapa 4: Causalidade de Granger e Correlação Cruzada (CCF)

### 1. Causalidade de Granger
Este teste não prova 'causa e efeito' no sentido filosófico, mas sim **precedência temporal**. Dizemos que $X$ causa $Y$ no sentido de Granger se:
$$Y_t \approx \alpha + \sum_{i=1}^k \beta_i Y_{t-i} + \sum_{i=1}^k \gamma_i X_{t-i}$$
Se os valores passados de $X$ ($IFBOI$) ajudam a prever o valor atual de $Y$ ($CEPEA$) melhor do que apenas o histórico do próprio $Y$, então o IFBOI possui **poder preditivo**.

### 2. Correlação Cruzada (CCF)
A CCF mede a correlação entre $X_{t-k}$ e $Y_t$ para diferentes atrasos ($k$).
- **Lag 0**: Correlação contemporânea (acontece no mesmo dia).
- **Lag 1, 2...**: Correlação com o IFBOI de 'n' dias atrás. Se o maior coeficiente estiver no Lag 1 ou 2, provamos que o mercado físico (CEPEA) demora a reagir à B3.

In [ ]:
# Correlação Cruzada Dinâmica
ccf_vals = sm.tsa.stattools.ccf(df_master['Ret_IFBOI'], df_master['Ret_CEPEA'])[:15]

fig_lags = go.Figure()
fig_lags.add_trace(go.Bar(
    x=[f'Lag {i}' for i in range(15)],
    y=ccf_vals,
    marker_color=['red' if x == max(ccf_vals) else 'royalblue' for x in ccf_vals]
))

fig_lags.update_layout(
    title='PODER PREDITIVO: Em qual dia o IFBOI mais influencia o CEPEA?',
    yaxis_title='Coeficiente de Correlação',
    template='plotly_white',
    annotations=[dict(x='Lag 1', y=max(ccf_vals), text='Pico de Influência', showarrow=True, arrowhead=1)]
)
fig_lags.show()

## Etapa 5: Síntese e Veredito Técnico

### 1. Consolidação Estatística
A análise conduzida confirma a hipótese de que o **IFBOI** não é apenas correlacionado ao **CEPEA**, mas atua como uma variável de sinalização antecedente. Os pilares que sustentam esta conclusão são:

*   **Estacionariedade Validada**: O teste ADF confirmou que a análise sobre retornos logarítmicos elimina o viés de tendência (inflação/ciclos), garantindo que a relação observada é estrutural e não espúria.
*   **Eficiência de Transmissão**: A Correlação Cruzada (CCF) identificou que o pico de influência ocorre com defasagem (*lag*), indicando que choques de preço na B3 levam tempo para serem totalmente absorvidos pelo mercado físico.
*   **Causalidade de Granger**: Estatisticamente, a série do IFBOI contém informações que reduzem o erro de previsão do CEPEA, validando seu papel como balizador de expectativas.

### 2. Recomendações Estratégicas (Executive Summary)
Para produtores, frigoríficos e investidores, os dados sugerem:
1.  **Hedge Antecipado**: Utilizar as variações do IFBOI como gatilho para operações de proteção em Bolsa, dado que o mercado físico tende a seguir o movimento com um atraso de 1 a 2 dias úteis.
2.  **Monitoramento de Volatilidade**: Períodos de alta volatilidade no IFBOI sem correspondência imediata no CEPEA devem ser interpretados como alertas de mudança iminente na tendência de preços do boi comum.
3.  **Arbitragem e Negociação**: Compradores e vendedores podem utilizar o 'spread' entre o índice futuro e o físico como métrica de poder de barganha nas negociações de balcão.

---
**Nota Final**: Este modelo demonstra que a integração financeira do mercado pecuário brasileiro permite uma gestão de risco baseada em evidências quantitativas, reduzindo a dependência de análises meramente subjetivas.